# 🚀 Your First LLM Call with Gemini

**Day 1 — Notebook 3 of 3 | Estimated Time: 45 minutes**

---

## What You'll Learn
- How to set up the Gemini API client
- How to make your first text generation call
- How to control generation with temperature, top-k, and top-p
- How to count tokens using the API

---

## 1. Setup

First, let's install the required package and set up authentication.

> 🔑 **Authentication:** You don't need an API key directly. Enter the course passphrase when prompted.

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os

# Add the repo root to the path so we can import our auth helper
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai

# Initialize the Gemini client
client = genai.Client(api_key=get_api_key())
print("✅ Gemini client initialized successfully!")

In [ ]:
# We'll use this model throughout the notebook
MODEL_ID = "gemini-2.5-flash"

---

## 2. Your First API Call

The simplest way to use Gemini is the `generate_content` method. You pass in a prompt (string) and get back a response.

In [ ]:
from IPython.display import Markdown

# Your very first LLM call!
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Explain what artificial intelligence is in 3 sentences.",
)

Markdown(response.text)

### Understanding the Response Object

The response contains more than just text. Let's explore it:

In [ ]:
# The response object has several useful attributes
print("Response text (first 200 chars):")
print(response.text[:200])
print()

# Token usage information
if response.usage_metadata:
    print("Token Usage:")
    print(f"  Input tokens:  {response.usage_metadata.prompt_token_count}")
    print(f"  Output tokens: {response.usage_metadata.candidates_token_count}")
    print(f"  Total tokens:  {response.usage_metadata.total_token_count}")

---

## 3. Multi-line Prompts

For longer prompts, use Python's triple-quoted strings. This is useful when you need to give the model detailed instructions.

In [ ]:
prompt = """
You are a helpful science teacher. 
Explain photosynthesis to a 10-year-old student.
Keep your explanation under 100 words.
Use simple language and a fun analogy.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)

Markdown(response.text)

---

## 4. Controlling Generation: Temperature

**Temperature** controls the randomness of the model's output:
- `temperature=0` → Deterministic, always picks the most likely token
- `temperature=1` → Balanced (default)
- `temperature=2` → Very creative/random

Let's see the difference:

In [ ]:
from google.genai import types

prompt = "Write a one-sentence description of the moon."

# Low temperature (more deterministic)
print("🧊 Temperature = 0 (deterministic)")
print("=" * 50)
for i in range(3):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0),
    )
    print(f"  Run {i+1}: {response.text.strip()}")

print()

# High temperature (more creative)
print("🔥 Temperature = 2 (creative)")
print("=" * 50)
for i in range(3):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=2),
    )
    print(f"  Run {i+1}: {response.text.strip()}")

> 💡 **Notice:** With temperature=0, the outputs are nearly identical. With temperature=2, each run produces different, more creative phrasing.

---

## 5. Controlling Generation: Top-K and Top-P

These parameters further control which tokens the model considers:

- **Top-K:** Only consider the K most likely tokens at each step
  - `top_k=1` → Only the single most likely token (very deterministic)
  - `top_k=40` → Consider top 40 tokens (more variety)

- **Top-P (nucleus sampling):** Consider tokens until cumulative probability reaches P
  - `top_p=0.1` → Only the most confident predictions
  - `top_p=0.95` → Consider a wider range of tokens

In [ ]:
prompt = "List 5 creative names for a coffee shop."

# Restrictive: low top-k, low top-p
print("🎯 Restrictive (top_k=1, top_p=0.1)")
print("=" * 50)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(top_k=1, top_p=0.1),
)
print(response.text)

print()

# Open: high top-k, high top-p, high temperature
print("🌈 Open (top_k=100, top_p=0.99, temperature=1.5)")
print("=" * 50)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(top_k=100, top_p=0.99, temperature=1.5),
)
print(response.text)

---

## 6. Controlling Output Length: Max Output Tokens

You can limit how many tokens the model generates with `max_output_tokens`:

In [ ]:
prompt = "Tell me the history of the internet."

# Short response
print("📝 Short (max_output_tokens=50)")
print("=" * 50)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(max_output_tokens=50),
)
print(response.text)
print(f"\n(Output tokens used: {response.usage_metadata.candidates_token_count})")

print()

# Longer response
print("📖 Longer (max_output_tokens=300)")
print("=" * 50)
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(max_output_tokens=300),
)
print(response.text)
print(f"\n(Output tokens used: {response.usage_metadata.candidates_token_count})")

---

## 7. Counting Tokens

You can count tokens before sending a request — useful for cost estimation and checking context window limits:

In [ ]:
# Count tokens for different texts
texts = [
    "Hello!",
    "The quick brown fox jumps over the lazy dog.",
    "Explain quantum computing in simple terms, covering superposition, entanglement, and quantum gates.",
]

print("Token Counting")
print("=" * 60)
for text in texts:
    token_count = client.models.count_tokens(
        model=MODEL_ID,
        contents=text,
    )
    words = len(text.split())
    print(f"\nText: \"{text[:60]}{'...' if len(text) > 60 else ''}\"")
    print(f"  Words:  {words}")
    print(f"  Tokens: {token_count.total_tokens}")
    print(f"  Ratio:  {token_count.total_tokens / words:.2f} tokens/word")

---

## 8. Listing Available Models

Let's see what Gemini models are available to us:

In [ ]:
# List available models
print("Available Gemini Models")
print("=" * 70)
for model in client.models.list():
    if "gemini" in model.name.lower():
        print(f"  {model.name}")

---

## 🏋️ Exercise 1: Your Turn — Ask Gemini Anything

Write a prompt of your choice and send it to Gemini. Try different types of tasks:
- Ask a factual question
- Ask it to write something creative
- Ask it to explain a concept

In [ ]:
# Exercise 1: Write your own prompt!
# TODO: Replace the prompt below with your own

my_prompt = "YOUR PROMPT HERE"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=my_prompt,
)

Markdown(response.text)

---

## 🏋️ Exercise 2: Temperature Experiment

Pick a creative writing prompt (e.g., "Write a haiku about coding"). Run it 3 times each at temperatures 0, 1, and 2. Observe how the outputs change.

In [ ]:
# Exercise 2: Temperature experiment
# TODO: Replace with your creative prompt

creative_prompt = "Write a haiku about coding"

for temp in [0, 1, 2]:
    print(f"\n{'='*50}")
    print(f"Temperature = {temp}")
    print(f"{'='*50}")
    for run in range(3):
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=creative_prompt,
            config=types.GenerateContentConfig(temperature=temp),
        )
        print(f"Run {run+1}: {response.text.strip()}")
        print()

---

## 🏋️ Exercise 3: Build a Simple Q&A Function

Create a reusable function that takes a question and returns Gemini's answer. Include token counting.

In [ ]:
# Exercise 3: Build a Q&A function

def ask_gemini(question: str, temperature: float = 1.0) -> str:
    """
    Send a question to Gemini and return the response.
    Also prints token usage information.
    
    Args:
        question: The question to ask
        temperature: Controls randomness (0-2)
    
    Returns:
        The model's response text
    """
    # TODO: Complete this function
    # 1. Call client.models.generate_content with the question and temperature
    # 2. Print the token usage (input, output, total)
    # 3. Return the response text
    
    pass  # Replace with your implementation


# Test your function
answer = ask_gemini("What are the three laws of thermodynamics?")
if answer:
    display(Markdown(answer))

---

## 🏋️ Exercise 4: Token Cost Calculator

Build a function that estimates the cost of an API call based on token count.

In [ ]:
# Exercise 4: Token cost calculator

# Approximate Gemini 2.5 Flash pricing (per 1M tokens)
INPUT_PRICE_PER_1M = 0.15   # $0.15 per 1M input tokens
OUTPUT_PRICE_PER_1M = 0.60  # $0.60 per 1M output tokens

def estimate_cost(prompt: str, expected_output_tokens: int = 500) -> dict:
    """
    Estimate the cost of an API call.
    
    Args:
        prompt: The input prompt
        expected_output_tokens: Estimated number of output tokens
    
    Returns:
        Dictionary with token counts and costs
    """
    # TODO: Complete this function
    # 1. Count input tokens using client.models.count_tokens()
    # 2. Calculate input cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_1M
    # 3. Calculate output cost = (expected_output_tokens / 1_000_000) * OUTPUT_PRICE_PER_1M
    # 4. Return a dict with all the info
    
    pass  # Replace with your implementation


# Test: estimate cost for a long prompt
test_prompt = "Write a detailed 1000-word essay about the history of artificial intelligence."
result = estimate_cost(test_prompt, expected_output_tokens=1500)
if result:
    print(f"Input tokens:  {result.get('input_tokens', 'N/A')}")
    print(f"Output tokens: {result.get('output_tokens', 'N/A')}")
    print(f"Input cost:    ${result.get('input_cost', 0):.6f}")
    print(f"Output cost:   ${result.get('output_cost', 0):.6f}")
    print(f"Total cost:    ${result.get('total_cost', 0):.6f}")

---

## 📝 Key Takeaways

1. **Gemini API** is accessed through the `google-genai` SDK using `genai.Client()`
2. **`generate_content()`** is the core method for text generation
3. **Temperature** controls randomness: 0 = deterministic, higher = more creative
4. **Top-K / Top-P** further control which tokens the model considers
5. **Token counting** helps estimate costs and check context window limits
6. Always monitor your **token usage** — it directly impacts cost

---

## 📚 Further Reading & Videos

| Resource | Type | Link |
|----------|------|------|
| Gemini API Quickstart | Docs | [ai.google.dev/gemini-api/docs/quickstart](https://ai.google.dev/gemini-api/docs/quickstart) |
| Gemini API Text Generation Guide | Docs | [ai.google.dev/gemini-api/docs/text-generation](https://ai.google.dev/gemini-api/docs/text-generation) |
| Understanding Temperature in LLMs | Article | [Towards Data Science](https://towardsdatascience.com/llm-temperature-parameter-demystified/) |
| Google GenAI SDK Reference | Docs | [googleapis.github.io/python-genai](https://googleapis.github.io/python-genai/) |

---

🎉 **Congratulations!** You've completed Day 1. You now understand what GenAI is, how LLMs work, and how to interact with Gemini via the API.

**Tomorrow:** [Day 2 — Prompt Engineering Fundamentals](../Day_02_Prompt_Engineering_Fundamentals/README.md)